# Lab 5: Structure from Motion

## Goals
The goal of the current assignment is to learn the following concepts:

- How to self-calibrate a camera using vanishing points.
- What are the main elements of an incremental structure from motion approach.

## Introduction

The Structure fron Motion problem (SfM) is defined as the 3D reconstruction from a set of unordered and uncalibrated images. There are different SfM approaches: global, hierarchical or incremental. In this lab we will focus on the incremental SfM, which is the most popular strategy for 3D reconstruction from unordered and uncalibrated photo collections.

Incremental SfM is a sequential processing pipeline with an iterative component. It starts working from two views, from which motion (camera parameters) are initialized, and then the structure (3D reconstruction) is also initialized. Right after, an iterative extension of both the motion and the structure is performed, progressively adding new cameras/views and 3D points. A further refinement of both the camera matrices and the reconstructed point cloud is carried out. This last step is known as Bundle Adjustment and it involves solving an optimization problem by iterative techniques.

The purpose of this lab is to familiarize with the structure from motion problem and work with the main blocks that constitute a basic (vanilla) incremental SfM algorithm, without the refinement step. A more complete and robust SfM pipeline would include -- apart from the bundle adjustment --  a carefully selection of the two initial views and each next new view to incorporate, filtering of outliers,  and some other tricks [3]. If you are interested in a more complete solution here we provide a list of some of the most known libraries and softwares:

- OpenMVG:  http://openmvg.readthedocs.io/en/latest/# <br>
Incremental and global SfM, open source.

- VisualSFM:  http://ccwu.me/vsfm/ <br>
Incremental SfM, very efficient, GUI, binaries.

- Bundler: http://www.cs.cornell.edu/~snavely/bundler/ <br>
Incremental SfM, open source.

- Colmap: http://colmap.github.io/ <br>
Incremental SfM, very efficient, nice GUI, open source.

- Theia: http://www.theia-sfm.org/ <br>
Incremental and Global SfM, very efficient, open source.

The solution of the structure from motion strongly relies on point correspondences (matchings) across the different views, commonly known as feature tracks. Then, one of the first things to do is feature extraction and matching, followed by geometric verification to remove outliers. 

As in previous labs, we will be using SIFT for estimating keypoints and matchings between pairs of images.  These matchings will contain outliers; these can be filtered by robustly estimating a fundamental matrix. Moreover, the fundamental matrix that relates the two initial views will be used to estimate the camera parameters (motion) of these two views.

In [ ]:
# !pip install requirements.txt

In [ ]:
import logging
import math
import random
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import scipy.io as sio
import seaborn_image as isns
from tqdm.notebook import tqdm

# `lab_3_utils.py` is YOUR module assembled from prior labs. Place a file with that
# name next to this notebook and expose the functions imported below (Normalization,
# fundamental_matrix and ransac_fundamental_matrix from Lab 3; decompose_essential,
# triangulate, single_resectioning and ransac_resectioning_adaptive_loop from Lab 4;
# the convenience wrapper `fundamental_and_inliers` is defined below in the notebook).
from lab_utils import plot_camera, plot_camera_col, reproj_err
from lab_3_utils import (
    Normalization,
    fundamental_matrix,
    ransac_fundamental_matrix,
    decompose_essential,
    triangulate,
    single_resectioning,
    ransac_resectioning_adaptive_loop,
)

# Deterministic seed so RANSAC inliers and the final cloud are reproducible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
cv2.setRNGSeed(SEED)


We will work with 3 different views of a scene from the UPF campus.

In [ ]:
# Read images
img1_color = cv2.imread(f"images/v1.jpg", cv2.IMREAD_COLOR)
img2_color = cv2.imread(f"images/v2.jpg", cv2.IMREAD_COLOR)
img3_color = cv2.imread(f"images/v3.jpg", cv2.IMREAD_COLOR)

img1_color = cv2.cvtColor(img1_color, cv2.COLOR_BGR2RGB)
img2_color = cv2.cvtColor(img2_color, cv2.COLOR_BGR2RGB)
img3_color = cv2.cvtColor(img3_color, cv2.COLOR_BGR2RGB)

# Reduce image size to speed up computations
original_shape = np.array(img1_color.shape[:2])
scale_percent = 50
rescaled_shape = np.flip(original_shape * scale_percent // 100)
img1r = cv2.resize(img1_color, rescaled_shape)
img2r = cv2.resize(img2_color, rescaled_shape)
img3r = cv2.resize(img3_color, rescaled_shape)

In [ ]:
g = isns.ImageGrid([img1_color, img2_color, img3_color], height=5, cmap="gray")
for i, axis in enumerate(g.axes[0], start=1):
    axis.set_title(f"View {i}")
plt.show()

## 1. Fundamental Matrix Estimation ($F_{12}$, $F_{13}$, $F_{23}$)

Robustly estimate the fundamental matrix for each of the three view pairs
(1–2, 1–3 and 2–3). For each pair, display the inlier matchings together
with the pair of images. $F_{23}$ is computed here as well so that the
strict three-view intersection in Section 5.2 can reuse its inliers
rather than re-running RANSAC later.

We recommend you to use SIFT with 13000 matches instead of 3000 which is the value in previous labs.

**Hint**: For a faster convergence of the RANSAC algorithm you can filter matched keypoints based on their matching distance. You can filter out matches that have a distance bigger than a threshold (e.g. 200).

In [ ]:
# TODO: Find keypoints

# Show matches
def plot_matches(img1, img2, kp1, kp2, matches, title="Matches"):
    """
    Plot matches between two images.
    """
    img_matches = cv2.drawMatches(
        img1,
        kp1,
        img2,
        kp2,
        matches,
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
    )
    plt.figure(figsize=(20, 10))
    plt.imshow(img_matches)
    plt.axis("off")
    plt.title(title)
    plt.show()
plot_matches(img1r, img2r, keypoints[0], keypoints[1], matches_12, "Matches 1-2")
plot_matches(img1r, img3r, keypoints[0], keypoints[2], matches_13, "Matches 1-3")
plot_matches(img2r, img3r, keypoints[1], keypoints[2], matches_23, "Matches 2-3")


In [ ]:
# TODO: Filter matches based on hint
# Complete ...

for name, before, after in [
    ("1-2", old_matches_12, matches_12),
    ("1-3", old_matches_13, matches_13),
    ("2-3", old_matches_23, matches_23),
]:
    print(f"Images {name} - {len(after)/len(before)*100:.2f}% of matches after filtering: {len(after)} of {len(before)}")

# Show filtered matches
plot_matches(img1r, img2r, keypoints[0], keypoints[1], matches_12, "Filtered Matches 1-2")
plot_matches(img1r, img3r, keypoints[0], keypoints[2], matches_13, "Filtered Matches 1-3")
plot_matches(img2r, img3r, keypoints[1], keypoints[2], matches_23, "Filtered Matches 2-3")


In [ ]:
# Wrap "DMatch list -> point arrays -> RANSAC F" so the three pair calls below
# stay short. ransac_fundamental_matrix is your Lab 3 RANSAC implementation.
def fundamental_and_inliers(matches, keypoints_a, keypoints_b,
                            th=5, min_iterations=1000):
    pts_a, pts_b = [], []
    for m in matches:
        pts_a.append((*keypoints_a[m.queryIdx].pt, 1.0))
        pts_b.append((*keypoints_b[m.trainIdx].pt, 1.0))
    pts_a = np.asarray(pts_a).T
    pts_b = np.asarray(pts_b).T

    F, ids = ransac_fundamental_matrix(pts_a, pts_b, th, min_iterations)
    inlier_matches = list(np.array(matches)[ids])
    return {
        "F": F,
        "inlier_ids": ids,
        "inliers": inlier_matches,
        "points1": pts_a,
        "points2": pts_b,
    }


# Run RANSAC-fundamental-matrix on each of the three pairs.
# fundamental_and_inliers wraps DMatch -> point arrays -> RANSAC -> dict.
print("=== Pair 1-2 ===")
res_12 = fundamental_and_inliers(matches_12, keypoints[0], keypoints[1])
print("=== Pair 1-3 ===")
res_13 = fundamental_and_inliers(matches_13, keypoints[0], keypoints[2])
print("=== Pair 2-3 ===")
res_23 = fundamental_and_inliers(matches_23, keypoints[1], keypoints[2])

# Unpack what downstream cells need
f_12, inliers_12, inlier_ids_12 = res_12["F"], res_12["inliers"], res_12["inlier_ids"]
f_13, inliers_13, inlier_ids_13 = res_13["F"], res_13["inliers"], res_13["inlier_ids"]
f_23, inliers_23, inlier_ids_23 = res_23["F"], res_23["inliers"], res_23["inlier_ids"]

points1_12, points2_12 = res_12["points1"], res_12["points2"]
points1_13, points3_13 = res_13["points1"], res_13["points2"]
points2_23, points3_23 = res_23["points1"], res_23["points2"]

f_matrices = [f_12, f_13, f_23]
inlier_matches = [inliers_12, inliers_13, inliers_23]


In [ ]:
# Show inlier matches for all three pairs
plot_matches(img1r, img2r, keypoints[0], keypoints[1], inliers_12, "Inlier Matches for F 1-2")
plot_matches(img1r, img3r, keypoints[0], keypoints[2], inliers_13, "Inlier Matches for F 1-3")
plot_matches(img2r, img3r, keypoints[1], keypoints[2], inliers_23, "Inlier Matches for F 2-3")


## 2. Self-Calibration via Vanishing Points ($K$)
In lab 4 we saw how to recover the motion between a pair of calibrated cameras. In SfM, cameras are not calibrated, but different self-calibration techniques can be applied for estimating the intrinsic parameters.
 
In case we work with images of man-made environments, like urban or indoor scenes, it is possible to estimate vanishing points of orthogonal directions. Vanishing points are useful for self-calibration because they allow us to establish constraints on the internal parameters of the camera (intrinsics). There are different methods in the literature to estimate vanishing points. For example, the following ones are implemented in Matlab:

- Vanishing Point Detection in Urban Scenes Using Point Alignments [1, 2]: <br>
http://www.ipol.im/pub/art/2017/148/

- Orthogonal Vanishing Points in Uncalibrated Images of Man-Made Environments [4]: <br>
https://members.loria.fr/GSimon/software/fastvp/

And this one in Python:
- NeurVPS: Neural Vanishing Point Scanning via Conic Convolution: [6] <br>
https://github.com/zhou13/neurvps

We will assume square pixels (aspect ratio of 1), zero skew factor and principal point at the center of the image, which is common in most commercial cameras.  Then, the only remaining unknown in the matrix of internal parameters is the focal length $\alpha$, but it can be estimated using a pair of vanishing points as we will explain now (see also Section 6.3.2 of Szeliski's book [5]). 

Under the previous assumptions, the camera calibration matrix, $K$, can be written as: <br>

$$ K=\begin{pmatrix} \alpha _x & s & c_x \\ 0 & \alpha _y & c_y \\ 0 & 0 & 1\end{pmatrix}
\overset{Assumptions}{\longrightarrow}
K=\begin{pmatrix} \alpha & 0 & c_x \\ 0 & \alpha & c_y \\ 0 & 0 & 1\end{pmatrix}$$
where the only unknown is $\alpha$, since $c_x = \frac{w}{2}$ and $c_y = \frac{h}{2}$, being $w$ and $h$, respectively, the image width and height in pixels. 

Let us assume that we have detected two or more orthogonal vanishing points, all of which are finite, i.e., they are not obtained from lines that appear to be parallel in the image plane. The projection equation for the vanishing point $\mathbf{v}_1$ corresponding to the cardinal direction $(1, 0, 0)^T$ can be written as
$$ \mathbf{v}_1 \sim P \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} = K  \begin{pmatrix}\mathbf{r}_1 & \mathbf{r}_2 & \mathbf{r}_3 & \mathbf{t} \end{pmatrix} \begin{pmatrix} 1 \\ 0 \\ 0 \\ 0 \end{pmatrix} = K \mathbf{r}_1$$
If we denote the coordinates of the vanishing point $\mathbf{v}_1$ as $x_1$ and $y_1$ we have:
$$ \mathbf{r}_1 \sim K^{-1} \mathbf{v}_1 = \begin{pmatrix} 1/\alpha & 0 & -c_x/\alpha \\ 0 & 1/\alpha & -c_y/\alpha \\ 0 & 0 & 1 \end{pmatrix} 
 \begin{pmatrix} x_1 \\ y_1 \\ 1 \end{pmatrix} = 
 \begin{pmatrix} (x_1 - c_x) / \alpha \\ (y_1 -c_y) / \alpha \\ 1 \end{pmatrix}
 \sim \begin{pmatrix} x_1 - c_x \\ y_1 -c_y \\ \alpha \end{pmatrix}$$
And in general, for the vanishing point $\mathbf{v}_i$, $i=1,2,3$, corresponding to one of the cardinal directions (1, 0, 0), (0, 1, 0), or (0, 0, 1) respectively, and $\mathbf{r}_i$ being the $i_{th}$ column of the rotation matrix $R$ we have:
$$ \mathbf{r}_i \sim  \begin{pmatrix} x_i - c_x \\ y_i -c_y \\ \alpha \end{pmatrix}$$
From the orthogonality between columns of the rotation matrix, we have:
$$ \mathbf{r}_i  \cdot \mathbf{r}_j = (x_i - c_x)(x_j - c_x)+(y_i - c_y)(y_j - c_y)+ \alpha^2 =0, $$
from which we can obtain an estimate for $\alpha$:
$$ \alpha = \sqrt{-(x_i - c_x)(x_j - c_x)-(y_i - c_y)(y_j - c_y)}.$$
Then, it is possible to estimate $\alpha$, and thus $K$, using two vanishing points corresponding to orthogonal directions. In our case, all the images have been taken with the same camera, so all of them will share the same $K$.

In this notebook we detect the vanishing points directly in Python using the Probabilistic Hough Transform (`cv2.HoughLinesP`): line segments are detected, all pairwise intersections are accumulated in a large grid (covering VPs far outside the image), and the pair of peaks that satisfies $\alpha^2 > 0$ is selected automatically.

Provide the code to estimate the matrix of internal parameters $K$ from the detected vanishing points. Which is the matrix you have obtained?

In [ ]:
# Vanishing points for this scene (manually picked for reproducibility).
vp1 = np.array([1932.51919443, 3033.59044871])
vp2 = np.array([1516.57688111, -14998.62215829])
vp1, vp2 = vp1 * scale_percent / 100, vp2 * scale_percent / 100
print(f"VP1 (calibration pair): [{vp1[0]:.1f}, {vp1[1]:.1f}]")
print(f"VP2 (calibration pair): [{vp2[0]:.1f}, {vp2[1]:.1f}]")


In [ ]:
# TODO: Complete
alpha = ...
K = ...

print(f"alpha: {alpha}")
print(f"Intrinsic matrix K:\n{K}")

**Optional theory.** 

Given what we have learned in slides 29-32 from lecture 5, we can find a different way to estimate $\alpha$. We know that $v_1^T \omega v_2 = 0$. We should explain that with the image of the absolute conic we can obtain the formula for the angle between 2 vectors. $cos(\theta) = \frac{v_1^T \omega v_2}{...}$, and knowing that $v_1$ and $v_2$ are orthogonal, we know that the numerator is equal to zero. Then, we know that $\omega = (K K^T)^-1$.

Let's understand the shape of $\omega$. We know that in the general case
$$ K = \begin{pmatrix} \alpha_x & s & c_x \\ 0 & \alpha_y & c_y \\ 0 & 0 & 1\end{pmatrix} $$
Then, the value of $\omega$ can be computed and the result is the following one (extracted from Zhang paper provided in lab 2):

$$\omega = \begin{pmatrix}
\frac{1}{\alpha_x^2} & - \frac{s}{\alpha_x^2\alpha_y} & \frac{c_y s - c_x \alpha_y}{\alpha_x^2\alpha_y} \\
- \frac{s}{\alpha_x^2\alpha_y} & \frac{s^2}{\alpha_x^2\alpha_y} + \frac{1}{\alpha_y^2} & - \frac{s (c_y s - c_x \alpha_y)}{\alpha_x^2\alpha_y} - \frac{c_y}{\alpha_y^2} \\
\frac{c_y s - c_x \alpha_y}{\alpha_x^2\alpha_y} & - \frac{s (c_y s - c_x \alpha_y)}{\alpha_x^2\alpha_y} - \frac{c_y}{\alpha_y^2} & \frac{(c_y s - c_x \alpha_y)^2}{\alpha_x^2\alpha_y^2} + \frac{c_y^2}{\alpha_y^2} + 1\end{pmatrix}
$$

Apply the assumptions of squared pixels and zero skew factor to demonstrate the same equation of $\alpha$ as in the previous explanation.

> Answer this **optional** derivation in the *Extras* section of the
> questionnaire. You can write it in LaTeX or attach a picture of your
> handwritten notes.


## 3. Initial Two-View Reconstruction ($P_1$, $P_2$ and 3D points $\mathbf{X}$)

In the first part of the lab we have estimated the fundamental matrix $F$ that relates the initial pair of views. With $F$ and $K$ we can estimate the essential matrix $E$ and from it we can get the complete camera matrices (intrinsics and extrinsics) for the initial pair of views. Once the cameras are fully calibrated an initial 3D reconstruction (structure) is found by triangulation. These steps were part of lab 3.

### 3.1 Camera matrices ($P_1$, $P_2$)

Estimate the camera matrices for views 1 and 2. 

In [ ]:
# TODO: Find P1
P1 = ...
print(f"P1: {P1}")

# TODO: Find Essential matrix for view 1-2
E_12 = ...
print(f"Essential matrix E_12:\n{E_12}")

# TODO: Find P2 by cheirality vote across the 4 candidate decompositions of E
P2 = ...
print(f"P2: {P2}")

# Show the final two cameras
fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
fig.update_layout(
    title="Camera Projection Matrices",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
)
fig.show()


### 3.2 Triangulation (3D points $\mathbf{X}$ from views 1 and 2)

Triangulate the matches from views 1 and 2 and plot them together with the cameras.

In [ ]:
# complete ...
# TODO: Triangulate all matches
X = ...

print(f"Number of points in the point cloud: {X.shape[1]}")


# TODO: Remove bad-triangulated points (Z < 0) for a proper visualization

# TODO: Filter on reprojection error (and trim the long depth tails)
# Most rays passing the F-RANSAC test still triangulate to absurd 3D
# positions when the baseline parallax is small. Reproject each 3D point
# in views 1 and 2 with reproj_err(P, X, x) and keep only those with
# error < 2 px in BOTH views, and depth Z in the [p5, p95] band.

print(f"Number of points in the point cloud after removing errors: {X.shape[1]}")

# Render the 3D point cloud
fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
x_img = inliers_1_12[:2].T.astype(int)
rgb_vals = img1r[x_img[:, 1], x_img[:, 0]]
rgb_vals = [f"rgb({int(r)},{int(g)},{int(b)})" for r, g, b in rgb_vals]
fig.add_trace(go.Scatter3d(x=X[0, :], y=X[2, :], z=-X[1, :], mode="markers", marker=dict(size=2, color=rgb_vals)))
fig.update_layout(
    title="3D Point Cloud - View 1-2",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
)
fig.show()


### 3.3 Reprojection error of the 1-2 cloud

As in Lab 3, look at the per-point reprojection error of the surviving
3D points to gauge how accurate the two-view reconstruction is. Plot a
histogram of the errors in views 1 and 2 (after the filter above) and
record the mean / median in the questionnaire.


In [ ]:
import seaborn as sns

# TODO: Compute the reprojection error of each 3D point in views 1 and 2
err1 = reproj_err(?, ?, ?)
err2 = reproj_err(?, ?, ?)

# Plot the histograms with mean / median markers
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, err, label in zip(axes, [err1, err2], ["view 1", "view 2"]):
    sns.histplot(err, bins=30, kde=True, ax=ax, color="steelblue")
    ax.axvline(err.mean(), color="red", ls="--",
               label=f"mean {err.mean():.2f} px")
    ax.axvline(np.median(err), color="orange", ls=":",
               label=f"median {np.median(err):.2f} px")
    ax.set_xlabel("reprojection error (px)")
    ax.set_title(f"Reprojection error - {label}")
    ax.legend()
plt.tight_layout()
plt.show()


## 4. Adding the Third Camera ($P_3$)

At this point we have reconstructed some 3D points from the point correspondences in the initial pair of views. If we are able to find a sufficient number of correspondences, in a new image, of the already reconstructed keypoints we will have a set of 3D-2D correspondences that can be used to calibrate the new view. For that, we will use the resectioning method (lecture 6) that needs  at least **six 3D-2D correspondences**. Alternatively, other methods, like $PnP$ approaches can be used for this purpose (since all cameras share the same matrix $K$).

We want to estimate $P_3$. Given $\vec{x_3}$ a set of keypoints from image 3, and $\vec{X_{1,2}}$ a set of 3D points extracted from the corresponding keypoints in image 1 and 2. We know that $\vec{x_3} \sim P_3 \vec{X_{1,2}}$. So, as $P_3$ has shape ($3x4$), it has 11 unkwons up to scale, and each correspondance will provide two equations, thus, we will need 6 correspondences.

### 4.1 2D–3D correspondences ($\mathbf{X} \leftrightarrow \mathbf{x}_3$)

**Find the 2D–3D correspondences**

1. Extract keypoint correspondences across all three images.
You already have matches for image pairs (1–2) and (1–3). Compute the intersection of these two match sets by comparing the indices of the keypoints in image 1. Call the resulting triplets of matching keypoints `kp1`, `kp2`, and `kp3`, corresponding to images 1, 2, and 3, respectively.

2. Form the 2D–3D correspondences.
The 2D points are the keypoints in image 3.
The 3D points come from triangulating each matching pair (kp1, kp2) using the known camera poses for images 1 and 2.

In [ ]:
# TODO: Find intersection matches between views 1-2 and 1-3
intersect_12, intersect_13 = [], []
# complete ...
kp1 = ...
kp2 = ...
kp3 = ...

assert len(kp1) == len(kp2) == len(kp3), "The number of points in all views should be the same."

# TODO: Define points_2d and points_3d
points_2d = ...
points_3d = ...

In [ ]:

# Print number of intersection matches and show matches on images
print(f"Number of intersection matches: {kp1.shape[1]} out of {len(matches_12)}")
plot_matches(img1r, img2r, keypoints[0], keypoints[1], intersect_12, "Intersection Matches 1–2")
plot_matches(img1r, img3r, keypoints[0], keypoints[2], intersect_13, "Intersection Matches 1–3")


### 4.2 Resectioning ($P_3$)

Create the function `resectioning` to calibrate the 3rd view and establish the proper entries to the function. Apply RANSAC to make it more robust.

In [ ]:
# TODO: Define resectioning function (uses RANSAC from lab_3_utils).
def resectioning(points_2d, points_3d): ...

# Find P3 by 2D-3D resectioning
P3_resectioned, inliers_3 = resectioning(points_2d, points_3d) ## P3_resectioned, inliers_3 = resectioning(points_2d, points_3d)
print(f"Number of inliers for P3: {len(inliers_3)}")


### 4.3 Camera decomposition ($K_3$, $R_3$, $\mathbf{t}_3$)

You will obtain $P_3$ by resectioning. However, in order to plot $P_3$, we will have to normalize it because the current scaling factor can be of any value. To do that we will decompose the projection matrix into $K_3$, $R_3$ and $t_3$. Normalize $K_3$ with its last element, dehomogenise $t_3$, and compute the new normalised projection matrix of camera 3, `P3_norm`.   

In [ ]:
# TODO: Decompose P3
K_3, R_3, t_3 = cv2.decomposeProjectionMatrix(P3)[:3]
K_3 /= ...
t_3 = t_3[:, 0] / ...
# t_3 from decomposeProjectionMatrix is the camera CENTRE C in world
# coordinates, not the extrinsic translation. P = K [R | -R C].
P3 = ??? @ np.column_stack((???, -??? @ ???))

print(f"Previous K:{K}", f"K_3: {K_3}", f"R_3: {R_3}", f"t_3: {t_3}", f"P3: {P3}", sep="\n")

In [ ]:
# Show the projection matrices
ny, nx, _ = img1r.shape
fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
plot_camera(P3, nx, ny, fig, "Camera 3")
fig.update_layout(
    title="Estimated Camera Projection Matrices",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
)
fig.show()

In [ ]:
# Visualize View 3 keypoints alongside the 3D intersection reconstruction

# 1) Image plane: image 3 with its matched (intersection) keypoints
fig_img3, ax_img3 = plt.subplots(figsize=(12, 8))
ax_img3.imshow(img3r)
ax_img3.scatter(kp3[0, :], kp3[1, :], s=8, c='red', linewidths=0)
ax_img3.set_title('Image 3 — intersection keypoints used for resectioning', fontsize=14)
ax_img3.axis('off')
plt.tight_layout()
plt.show()

# 2) 3D scene: triangulated intersection points coloured from image 3
X_intersection = points_3d[:-1] / points_3d[-1]  # 3×N Euclidean
kp3_int = kp3[:2].T.astype(int).clip([0, 0], [img3r.shape[1] - 1, img3r.shape[0] - 1])
rgb_vals_intersection = img3r[kp3_int[:, 1], kp3_int[:, 0]]
rgb_vals_intersection = [f'rgb({int(r)},{int(g)},{int(b)})' for r, g, b in rgb_vals_intersection]

fig_3d_intersection = go.Figure()
plot_camera(P1, nx, ny, fig_3d_intersection, 'Camera 1')
plot_camera(P2, nx, ny, fig_3d_intersection, 'Camera 2')
plot_camera(P3, nx, ny, fig_3d_intersection, 'Camera 3')
fig_3d_intersection.add_trace(go.Scatter3d(
    x=X_intersection[0, :], y=X_intersection[2, :], z=-X_intersection[1, :],
    mode='markers', marker=dict(size=2, color=rgb_vals_intersection)
))
fig_3d_intersection.update_layout(
    title='3D Intersection Points with All 3 Cameras',
    scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z', aspectmode='data'),
)
fig_3d_intersection.show()

### 4.4 Qualitative view (all cameras + 3D cloud)

Now, let's show the 3D point clouds with all the cameras.

In [ ]:
ny, nx, _ = img1r.shape
fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
plot_camera(P3, nx, ny, fig, "Camera 3")
fig.add_trace(go.Scatter3d(x=X[0, :], y=X[2, :], z=-X[1, :], mode="markers", marker=dict(size=2, color=rgb_vals)))
fig.update_layout(
    title="Estimated 3D Points and Camera Projection Matrices",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        aspectmode="data"
    ),
)
fig.show()

## 5. Cloud Densification and Evaluation

### 5.1 Densify the cloud (new $\mathbf{X}$ from $F_{13}$ inliers)

Triangulate the matches from views 1 and 3 to add new 3D points to the
cloud you obtained from views 1 and 2. You may use `cv2.findFundamentalMat`
in place of the custom RANSAC if you wish.


In [ ]:
# TODO: triangulate F_13 inliers with P1 and P3,
# filter the same way as the 1-2 cloud, and add to the point cloud.

fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
plot_camera(P3, nx, ny, fig, "Camera 3")
fig.add_trace(go.Scatter3d(
    x=X_full[0, :], y=X_full[2, :], z=-X_full[1, :],
    mode="markers", marker=dict(size=2, color=rgb_full)))
fig.update_layout(
    title="Densified 3D point cloud (1-2 ∪ 1-3) with all 3 cameras",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
)
fig.show()


### 5.2 Strict three-view intersection ($F_{12} \cap F_{13} \cap F_{23}$ inliers)

Instead of the union of inliers from views 1–2 and 1–3, keep only points
that are inliers across **all three** pairs (1–2, 1–3, 2–3). This guarantees
every kept 3D point is consistent with the epipolar geometry of every
camera that saw it, at the cost of a smaller cloud. We already estimated
$F_{23}$ in Section 1, so here we just chain `inliers_12`, `inliers_13`
and `inliers_23` through keypoint indices.


In [ ]:
# TODO: triangulate only the matches that are RANSAC inliers
# in all three view pairs (1-2, 1-3, 2-3).

fig = go.Figure()
plot_camera(P1, nx, ny, fig, "Camera 1")
plot_camera(P2, nx, ny, fig, "Camera 2")
plot_camera(P3, nx, ny, fig, "Camera 3")
fig.add_trace(go.Scatter3d(
    x=X_tri[0, :], y=X_tri[2, :], z=-X_tri[1, :],
    mode="markers", marker=dict(size=2, color=rgb_tri)))
fig.update_layout(
    title="Strict-intersection cloud (inliers in 1-2 ∩ 1-3 ∩ 2-3)",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
)
fig.show()


### 5.3 Quantitative evaluation of the three clouds

So far each cloud was inspected only visually. Let's quantify the trade-off:
for each pair we report the number of RANSAC inliers (out of the filtered
matches), and for each cloud we report the number of 3D points that survived
outlier rejection and the mean / median reprojection error in every view that
saw them.


In [ ]:
# Quantitative comparison of the three clouds
import numpy as np

def _stats(P, X, x):
    """Mean and median reprojection error of 3D points X in camera P, against 2D x."""
    e = reproj_err(P, X, x)
    return e.mean(), np.median(e)

# --- Epipolar geometry: matches and RANSAC inliers ---
print("Epipolar geometry")
for name, n_match, n_in in [
    ("1-2", len(matches_12), len(inliers_12)),
    ("1-3", len(matches_13), len(inliers_13)),
    ("2-3", len(matches_23), len(inliers_23)),
]:
    print(f"  F_{name}: {n_in:5d} inliers / {n_match:5d} matches "
          f"({100*n_in/n_match:5.1f}%)")

# --- Cloud sizes and reprojection error ---
print("\nClouds (after outlier rejection)")
rows = []

# Initial 1-2 cloud
m1, md1 = _stats(P1, X, inliers_1_12)
m2, md2 = _stats(P2, X, inliers_2_12)
rows.append(("initial 1-2", X.shape[1], (m1, m2, None), (md1, md2, None)))

# Added 1-3 sub-cloud
m1, md1 = _stats(P1, X_13, inliers_1_13)
m3, md3 = _stats(P3, X_13, inliers_3_13)
rows.append(("added 1-3", X_13.shape[1], (m1, None, m3), (md1, None, md3)))

# Densified union
rows.append(("densified 1-2 \u222a 1-3", X_full.shape[1], (None, None, None), (None, None, None)))

# Strict three-view intersection
m1, md1 = _stats(P1, X_tri, tri_pts1)
m2, md2 = _stats(P2, X_tri, tri_pts2)
m3, md3 = _stats(P3, X_tri, tri_pts3)
rows.append(("strict 1-2 \u2229 1-3 \u2229 2-3", X_tri.shape[1],
             (m1, m2, m3), (md1, md2, md3)))

header = f"  {'cloud':<28}{'N pts':>8}   {'mean err (px) 1/2/3':>26}   {'median err (px) 1/2/3':>26}"
print(header)
for name, npts, means, meds in rows:
    def fmt(t):
        return "/".join(f"{v:5.2f}" if v is not None else "  -  " for v in t)
    print(f"  {name:<28}{npts:>8}   {fmt(means):>26}   {fmt(meds):>26}")


## 6. (**EXTRA**) Improve the previous method
Suggest an improvement and try it.

## Lab References

[1] José Lezama, Rafael Grompone von Gioi, Gregory Randall, and Jean-Michel Morel. Finding vanishing points via point alignments in image primal and dual domains. In Proceedings of the IEEE Conference on Computer Vision and Pattern Recognition, pages 509–515, 2014.

[2] José Lezama, Gregory Randall, and Rafael Grompone von Gioi. Vanishing Point Detection in Urban Scenes Using Point Alignments. Image Processing On Line, 7:131–164, 2017.

[3] Johannes Lutz Schönberger and Jan-Michael Frahm. Structure-from-motion revisited. In Conference on Computer Vision and Pattern Recognition (CVPR), 2016.

[4] Gilles Simon, Antoine Fond, and Marie-Odile Berger. A simple and effective method to detect orthogonal vanishing points in uncalibrated images of man-made environments. In Eurographics, 2016.

[5] Richard Szeliski. Computer vision: algorithms and applications. Springer Science & Business Media, 2010.

[6] Yichao Zhou, Haozhi Qi, Jingwei Huang, and Yi Ma. Neurvps: Neural vanishing point scanning via conic convolution. NeurIPS 2019. 